In [1]:
import json
import pandas as pd
import re
import os
import argparse
from typing import Dict, List, Any

# 提取诊断结果的函数
def extract_diagnosis_result(diagnosis_text: str) -> str:
    """
    从诊断文本中提取<box></box>中的诊断结果
    """
    # 查找 <box>xxx</box> 格式的诊断结果
    pattern = r'<box>([^<]+)</box>'
    match = re.search(pattern, diagnosis_text)
    if match:
        return match.group(1).strip()
    return "others"  # 如果没有找到，返回默认值

# ICD代码到新标签的映射函数
def map_icd_to_new_label(icd_code: str) -> str:
    """
    将原来的ICD代码映射到新的诊断标签
    """
    mapping = {
        "F32": "抑郁",
        "F41": "焦虑", 
        "F32,F41": "mix",
        "Others": "others"
    }
    return mapping.get(icd_code, "others")

# 从原始回答中提取ICD代码并映射到新标签
def extract_and_map_ground_truth(gpt_content: str) -> str:
    """
    从GPT回答中提取ICD代码并映射到新的诊断标签
    如果回答中包含新格式的<box>标签，直接提取；否则从ICD代码映射
    """
    # 首先尝试提取新格式的<box>标签
    box_result = extract_diagnosis_result(gpt_content)
    if box_result != "others":  # 如果找到了有效的box标签
        return box_result
    
    # 如果没有找到box标签，尝试从ICD代码映射
    icd_pattern = r'\\icd_code\{([^}]+)\}'
    icd_match = re.search(icd_pattern, gpt_content)
    if icd_match:
        icd_code = icd_match.group(1)
        return map_icd_to_new_label(icd_code)
    
    return "others"

# 数据转换函数
def make_map_fn(data_source: str, split: str):
    """
    创建数据转换函数
    """
    def process_fn(data_item: Dict[str, Any], idx: int) -> Dict[str, Any]:
        visit_number = data_item['visit_number']
        conversations = data_item['conversations']
        
        # 提取ground truth并验证是否为空
        ground_truth = data_item.get("answer", [])
        
        # 过滤掉ground_truth为空的数据
        if not ground_truth or (isinstance(ground_truth, list) and len(ground_truth) == 0):
            return None  # 返回None表示跳过此条数据
        
        # 新的system和user prompt
        system_prompt = """你是一位经验丰富的精神科医生。请阅读以下初次精神科门诊的问诊对话记录，并根据ICD-10国际疾病分类标准，仔细分析后输出患者诊断结束后的ICD-10诊断代码。

## 注意：
1. 问诊对话为初次问诊，在症状严重程度和细节不可判断的时候，请推荐未特指的icd code。
2. 诊断结果可能包含1至2个icd-10诊断结果，大多只包含一个但不超过2个。
3. 用分号分隔不同的代码。
4. 需要严格根据icd-10标准来进行诊断的分析, 避免猜测和无根据的诊断，避免诊断错误。"""
        
        human_content = ""
        gpt_content = ""
        
        for conv in conversations:
            if conv['from'] == 'human':
                human_content = conv['value']
            elif conv['from'] == 'gpt':
                gpt_content = conv['value']
        
        # 提取对话文本（去掉原有的prompt前缀）
        text = human_content
        # 提取[问诊对话开始]到[问诊对话结束]之间的内容
        start_marker = "[问诊对话开始]"
        end_marker = "[问诊对话结束]"
        start_idx = text.find(start_marker)
        end_idx = text.find(end_marker)
        if start_idx != -1 and end_idx != -1:
            text = text[start_idx + len(start_marker):end_idx].strip()
        
        user_prompt = f"""
[问诊对话开始]
{text}
[问诊对话结束]

## 输出格式：
请用中文一步一步思考，并将思考过程放在<think>xxx</think>中输出，之后将最后诊断的ICD-10代码必须放在<box>xxx</box>中输出，用分号分隔，格式如：<think>xxx</think><box>Fxx.x;Fxx.x;Fxx.x</box>。"""
        
        # 构建prompt - 按照HuggingFace chat template格式
        prompt = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user", 
                "content": user_prompt
            }
        ]
        
        # 提取ground truth（映射到新标签）
        ground_truth = data_item["answer"]
        full_icd_code = data_item["ground_truth_codes"]
        
        # 构建转换后的数据
        converted_data = {
            "data_source": data_source,
            "prompt": prompt,
            "ability": "medical_diagnosis",  # 医疗诊断能力
            "reward_model": {
                "style": "rule",
                "ground_truth": ground_truth
            },
            "extra_info": {
                "split": split,
                "index": idx,
                "visit_number": visit_number,
                "original_response": gpt_content,
                "full_icd_code": full_icd_code
            }
        }
        
        return converted_data
    
    return process_fn

def validate_files(files: List[str]) -> List[str]:
    """
    验证文件列表，移除不存在的文件
    """
    valid_files = []
    for file_path in files:
        if os.path.exists(file_path):
            valid_files.append(file_path)
            print(f"✓ 文件验证通过: {file_path}")
        else:
            print(f"✗ 文件不存在，已跳过: {file_path}")
    return valid_files

def remove_duplicates_by_visit_number(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    根据visit_number去除重复数据，保留第一次出现的数据
    """
    seen_ids = set()
    deduplicated_data = []
    
    for item in data:
        visit_number = item.get('visit_number', '')
        if visit_number not in seen_ids:
            seen_ids.add(visit_number)
            deduplicated_data.append(item)
        else:
            print(f"发现重复数据，已跳过: {visit_number}")
    
    return deduplicated_data

def convert_sharegpt_to_parquet(train_files: List[str], val_files: List[str], output_dir: str, 
                               remove_duplicates: bool = True):
    """
    将ShareGPT格式的数据转换为后训练格式的parquet文件
    支持多个训练文件和验证文件输入
    
    Args:
        train_files: 训练数据文件路径列表
        val_files: 验证数据文件路径列表 
        output_dir: 输出目录
        remove_duplicates: 是否根据visit_number去除重复数据
    """
    data_source = "SMHC_RL_auxiliary_diagnosis"
    
    # 验证文件存在性
    print("正在验证文件...")
    train_files = validate_files(train_files)
    val_files = validate_files(val_files)
    
    if not train_files:
        raise ValueError("没有找到有效的训练文件")
    if not val_files:
        raise ValueError("没有找到有效的验证文件")
    
    # 读取多个训练数据文件
    train_data = []
    total_train_items = 0
    for train_file in train_files:
        print(f"正在读取训练数据: {train_file}")
        with open(train_file, 'r', encoding='utf-8') as f:
            file_data = json.load(f)
            train_data.extend(file_data)
            total_train_items += len(file_data)
            print(f"  读取到 {len(file_data)} 条数据")
    
    # 读取多个验证数据文件
    val_data = []
    total_val_items = 0
    for val_file in val_files:
        print(f"正在读取验证数据: {val_file}")
        with open(val_file, 'r', encoding='utf-8') as f:
            file_data = json.load(f)
            val_data.extend(file_data)
            total_val_items += len(file_data)
            print(f"  读取到 {len(file_data)} 条数据")
    
    print(f"\n合并后数据统计:")
    print(f"训练数据: {total_train_items} 条 (来自 {len(train_files)} 个文件)")
    print(f"验证数据: {total_val_items} 条 (来自 {len(val_files)} 个文件)")
    
    # 去重处理
    if remove_duplicates:
        print("\n正在进行数据去重...")
        original_train_count = len(train_data)
        original_val_count = len(val_data)
        
        train_data = remove_duplicates_by_visit_number(train_data)
        val_data = remove_duplicates_by_visit_number(val_data)
        
        print(f"训练数据去重: {original_train_count} -> {len(train_data)} (移除 {original_train_count - len(train_data)} 条)")
        print(f"验证数据去重: {original_val_count} -> {len(val_data)} (移除 {original_val_count - len(val_data)} 条)")
    
    # 创建转换函数
    train_map_fn = make_map_fn(data_source, 'train')
    val_map_fn = make_map_fn(data_source, 'val')
    
    # 转换数据
    print("正在转换训练数据...")
    converted_train = []
    filtered_train_count = 0
    for idx, item in enumerate(train_data):
        result = train_map_fn(item, idx)
        if result is not None:
            converted_train.append(result)
        else:
            filtered_train_count += 1
    
    print("正在转换验证数据...")
    converted_val = []
    filtered_val_count = 0
    for idx, item in enumerate(val_data):
        result = val_map_fn(item, idx)
        if result is not None:
            converted_val.append(result)
        else:
            filtered_val_count += 1
    
    # 输出过滤统计信息
    print(f"训练数据过滤统计: 原始 {len(train_data)} 条，过滤掉 {filtered_train_count} 条空ground_truth，保留 {len(converted_train)} 条")
    print(f"验证数据过滤统计: 原始 {len(val_data)} 条，过滤掉 {filtered_val_count} 条空ground_truth，保留 {len(converted_val)} 条")
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 转换为DataFrame并保存为parquet
    print("正在保存为parquet格式...")
    train_df = pd.DataFrame(converted_train)
    val_df = pd.DataFrame(converted_val)
    
    train_output_path = os.path.join(output_dir, 'train.parquet')
    val_output_path = os.path.join(output_dir, 'val.parquet')
    
    train_df.to_parquet(train_output_path, index=False)
    val_df.to_parquet(val_output_path, index=False)
    
    print(f"训练数据已保存至: {train_output_path}")
    print(f"验证数据已保存至: {val_output_path}")
    print(f"训练数据量: {len(converted_train)} 条")
    print(f"验证数据量: {len(converted_val)} 条")
    
    # 显示样例数据
    print("\n样例数据 (训练集第一条):")
    print(str(json.dumps(converted_train[0], ensure_ascii=False, indent=2)))
    
    return train_df, val_df

def add_multiple_files_example():
    """
    示例：如何添加多个文件
    """
    # 方式1：直接列出所有文件
    train_files = [
        "/tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-train-sharegpt.json",
        "/tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-asr-train-sharegpt.json",
        # "/tcci_mnt/shihao/project/LLaMA-Factory/shihao/data/mirodiag-16k-kimi-k2-0905-rl-train-sharegpt.json",
        # "/path/to/additional_train_file1.json",
        # "/path/to/additional_train_file2.json",
    ]
    
    val_files = [
        "/tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-test-sharegpt.json",
        # "/tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-asr-test-sharegpt.json",
        # "/path/to/additional_val_file1.json", 
        # "/path/to/additional_val_file2.json",
    ]
    
    return train_files, val_files

def get_files_from_pattern(pattern: str) -> List[str]:
    """
    根据文件模式获取文件列表
    """
    import glob
    files = glob.glob(pattern)
    files.sort()  # 确保文件顺序一致
    return files

# 设置文件路径 - 支持多文件输入
train_files, val_files = add_multiple_files_example()

# 你也可以使用文件模式匹配（如果有多个相似命名的文件）
# train_files = get_files_from_pattern("/path/to/train_data_*.json")
# val_files = get_files_from_pattern("/path/to/val_data_*.json")

output_dir = "/tcci_mnt/shihao/project/verl/psy_r1/SMHC-real_data_v7"

# 配置选项
REMOVE_DUPLICATES = False  # 是否去除重复数据

# 执行转换（带过滤功能）
print("开始执行完整数据转换（已添加ground_truth过滤）...")
print(f"训练文件数量: {len(train_files)}")
print(f"验证文件数量: {len(val_files)}")
print(f"去重选项: {'启用' if REMOVE_DUPLICATES else '禁用'}")
print(f"过滤空ground_truth: 启用")
print("-" * 50)

train_df, val_df = convert_sharegpt_to_parquet(
    train_files=train_files, 
    val_files=val_files, 
    output_dir=output_dir,
    remove_duplicates=REMOVE_DUPLICATES
)


开始执行完整数据转换（已添加ground_truth过滤）...
训练文件数量: 2
验证文件数量: 1
去重选项: 禁用
过滤空ground_truth: 启用
--------------------------------------------------
正在验证文件...
✓ 文件验证通过: /tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-train-sharegpt.json
✓ 文件验证通过: /tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-asr-train-sharegpt.json
✓ 文件验证通过: /tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-test-sharegpt.json
正在读取训练数据: /tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-train-sharegpt.json
  读取到 2217 条数据
正在读取训练数据: /tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-asr-train-sharegpt.json
  读取到 2206 条数据
正在读取验证数据: /tcci_mnt/shihao/data/rl_recommendation_dataset/smhc-recommendation-test-sharegpt.json
  读取到 596 条数据

合并后数据统计:
训练数据: 4423 条 (来自 2 个文件)
验证数据: 596 条 (来自 1 个文件)
正在转换训练数据...
正在转换验证数据...
训练数据过滤统计: 原始 4423 条，过滤掉 0 条空ground_truth，保留 4423 条
验证数据过滤统计: 原始 596 条，过滤掉 0 条空ground_truth，保留 596 条
正在保存为parquet格式...
训练数据已保存至: /tcci_mnt/

In [2]:
# 验证和使用生成的parquet文件

# 1. 读取parquet文件
import pandas as pd
import json

# 读取生成的parquet文件
train_df = pd.read_parquet('/tcci_mnt/shihao/project/verl/psy_r1/SMHC-MiroDiag_data_v7/train.parquet')
val_df = pd.read_parquet('/tcci_mnt/shihao/project/verl/psy_r1/SMHC-MiroDiag_data_v7/val.parquet')

print(f"训练集数据量: {len(train_df)}")
print(f"验证集数据量: {len(val_df)}")
print("\n数据列名:")
print(train_df.columns.tolist())

# 2. 检查数据结构
print("\n数据样例:")
sample = train_df.iloc[0]

# 安全地显示样例数据，避免numpy数组序列化问题
print("第一条数据的结构:")
print(f"- data_source: {sample['data_source']}")
print(f"- ability: {sample['ability']}")
print(f"- prompt: 包含 {len(sample['prompt'])} 个消息")
print(f"- reward_model: {sample['reward_model']}")
print(f"- extra_info: {sample['extra_info']}")

# 显示prompt的内容（截取前200个字符）
print(f"\n- prompt内容预览:")
prompt_content = sample['prompt'][0]['content'][:200] + "..." if len(sample['prompt'][0]['content']) > 200 else sample['prompt'][0]['content']
print(f"  角色: {sample['prompt'][0]['role']}")
print(f"  内容: {prompt_content}")

# 3. 统计不同诊断类别的分布
print("\n诊断类别分布:")

# 训练集诊断分布
train_diagnosis = train_df['reward_model'].apply(lambda x: x['ground_truth'])
print("训练集:")
print(train_diagnosis.value_counts())

# 验证集诊断分布
val_diagnosis = val_df['reward_model'].apply(lambda x: x['ground_truth'])
print("\n验证集:")
print(val_diagnosis.value_counts())

# 4. 检查数据质量
print("\n数据质量检查:")
print(f"训练集空值检查: {train_df.isnull().sum().sum()}")
print(f"验证集空值检查: {val_df.isnull().sum().sum()}")

# 5. 示例：如何使用数据进行后训练
print("\n后训练数据使用示例:")
print("每条数据包含以下字段:")
print("- data_source: 数据来源")
print("- prompt: HuggingFace chat template格式的对话")
print("- ability: 任务类型 (medical_diagnosis)")
print("- reward_model: 包含ground_truth的奖励模型信息")
print("- extra_info: 额外信息 (split, index, visit_number, original_response)")

print("\n转换完成！数据已准备好用于后训练。")


训练集数据量: 44948
验证集数据量: 596

数据列名:
['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']

数据样例:
第一条数据的结构:
- data_source: SMHC_RL_auxiliary_diagnosis
- ability: medical_diagnosis
- prompt: 包含 2 个消息
- reward_model: {'ground_truth': array(['F30'], dtype=object), 'style': 'rule'}
- extra_info: {'full_icd_code': array(['F30.100x001'], dtype=object), 'index': 0, 'original_response': '', 'split': 'train', 'visit_number': '206155103'}

- prompt内容预览:
  角色: system
  内容: 你是一位经验丰富的精神科医生。请阅读以下初次精神科门诊的问诊对话记录，并根据ICD-10国际疾病分类标准，仔细分析后输出患者诊断结束后的ICD-10诊断代码。

## 注意：
1. 问诊对话为初次问诊，在症状严重程度和细节不可判断的时候，请推荐未特指的icd code。
2. 诊断结果可能包含1至2个icd-10诊断结果，大多只包含一个但不超过2个。
3. 用分号分隔不同的代码。
4. 需要严格根...

诊断类别分布:
训练集:
reward_model
[F41]         20845
[F32]          7470
[F39]          4406
[F98]          1987
[F51]          1236
              ...  
[F32, F41]        1
[F32, F41]        1
[F32, F41]        1
[F32, F41]        1
[F41, F42]        1
Name: count, Length: 6518, dtype: int64

验证集:
reward_model
[F32]         